In [ ]:
# Block 1: notebook description and analysis objective

# This notebook isolates single-asset factor attribution and idiosyncratic risk.
# Extracted from Momentum & Efficiency.ipynb.

In [ ]:
import logging
import warnings
from pathlib import Path
import sys

import plotly.io as pio
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


from Quantapp.visualization.views.single_asset_profile.pricing.factor_analysis import (
    plot_idiosyncratic_risk_view,
    plot_rolling_regression_view,
)
from Quantapp.models import FactorRegressionModel

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")

In [ ]:
logger.disabled = True
logger.propagate = False

factor_model = FactorRegressionModel()

PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)

In [ ]:
# Block 3: set notebook parameters

factor_params = {
    "ticker_str": "SOXL",
    "interval": "1d",
    "analysis_period": "max",
    "rolling_window": 252,
    "ff_verbose": True,
}

ticker_str = factor_params["ticker_str"]
interval = factor_params["interval"]
analysis_period = factor_params["analysis_period"]
rolling_window = factor_params["rolling_window"]
ff_verbose = factor_params["ff_verbose"]

factor_params

In [ ]:
# Block 4: run factor attribution and idiosyncratic risk analysis
ff_proxy = factor_model.run_ff5_proxy_analysis(
    asset_ticker=ticker_str,
    period=analysis_period,
    interval=interval,
    window=rolling_window,
    auto_window=True,
    verbose=ff_verbose,
)

prices_df = ff_proxy["proxy_prices"]
returns = ff_proxy["proxy_returns"]
factor_returns = ff_proxy["factor_returns_all"]
factor_returns_capm = ff_proxy["factor_returns_capm"]
factor_returns_ff3 = ff_proxy["factor_returns_ff3"]
factor_returns_ff5 = ff_proxy["factor_returns_ff5"]
factor_returns_ff5_custom = factor_returns_ff5.copy()
stock_returns = ff_proxy["stock_returns"]
rolling_results_ff5_custom = ff_proxy["rolling_results"]

# Plot the Fama-French factor analysis results
ff_figs = plot_rolling_regression_view(
    rolling_results_ff5_custom,
    ticker_str,
    factor_returns_ff5_custom,
)
show_plotly_figure(ff_figs["alpha"])
show_plotly_figure(ff_figs["betas"])
show_plotly_figure(ff_figs["r_squared"])

idio_fig = plot_idiosyncratic_risk_view(
    rolling_results_ff5_custom,
    ticker_str,
    template="plotly_dark",
)
show_plotly_figure(idio_fig)